In [14]:
import os

files = os.listdir('../data/raw/county_health_rankings')
for f in sorted(files):
    print(f)

2016 County Health Rankings California Data - v3.xls
2017 County Health Rankings California Data - v2.xls
2018 County Health Rankings California Data - v3.xls
2019 County Health Rankings California Data - v1_0.xls
2020 County Health Rankings California Data - v1_0.xlsx
2021 County Health Rankings California Data - v1.xlsx
2022 County Health Rankings California Data - v2.xlsx
2023 County Health Rankings California Data - v3.xlsx
2024 County Health Rankings California Data - v2.xlsx
2025 County Health Rankings California Data - v3.xlsx


In [15]:
import pandas as pd
import numpy as np
import os

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

Load the 2024 CHR file to inspect structure and confirm availability of outcome 
variables (Premature Death, Preventable Hospital Stays, Poor Mental Health Days) 
that we will later regress on Median Multiple, our measure of housing unaffordability.

In [16]:
# Load 2024 CHR file
file_path = '../data/raw/county_health_rankings/2024 County Health Rankings California Data - v2.xlsx'

df_2024 = pd.read_excel(
    file_path,
    sheet_name='Additional Measure Data',
    header=[0, 1]  # Multi-row header based on what we saw
)

# Inspect
print('Shape:', df_2024.shape)
print('\nColumn names:')
for col in df_2024.columns:
    print(col)

Shape: (59, 344)

Column names:
('Unnamed: 0_level_0', 'FIPS')
('Unnamed: 1_level_0', 'State')
('Unnamed: 2_level_0', 'County')
('Life Expectancy', 'Life Expectancy')
('Life Expectancy', '95% CI - Low')
('Life Expectancy', '95% CI - High')
('Life Expectancy', 'Life Expectancy (Hispanic (all races))')
('Life Expectancy', 'Life Expectancy (Hispanic (all races)) 95% CI - Low')
('Life Expectancy', 'Life Expectancy (Hispanic (all races)) 95% CI - High')
('Life Expectancy', 'Life Expectancy (Non-Hispanic AIAN)')
('Life Expectancy', 'Life Expectancy (Non-Hispanic AIAN) 95% CI - Low')
('Life Expectancy', 'Life Expectancy (Non-Hispanic AIAN) 95% CI - High')
('Life Expectancy', 'Life Expectancy (Non-Hispanic Asian)')
('Life Expectancy', 'Life Expectancy (Non-Hispanic Asian) 95% CI - Low')
('Life Expectancy', 'Life Expectancy (Non-Hispanic Asian) 95% CI - High')
('Life Expectancy', 'Life Expectancy (Non-Hispanic Black)')
('Life Expectancy', 'Life Expectancy (Non-Hispanic Black) 95% CI - Low')
('L

Of the potential outcome variables available in the 'Additional Measure Data' 
sheet, Life Expectancy was considered but rejected in favor of Premature Death 
(YPLL — Years of Potential Life Lost), available in the 'Select Measure Data' 
sheet. YPLL weights early deaths more heavily than average life expectancy, 
making it more sensitive to the effects of housing unaffordability and economic 
stress on working-age adults.

From the 'Select Measure Data' sheet, three outcome variables were selected 
for regression against the Median Multiple, our primary measure of housing 
unaffordability: Premature Death (Years of Potential Life Lost Rate), 
Preventable Hospital Stays (Preventable Hospitalization Rate), and Poor 
Mental Health Days (Average Number of Mentally Unhealthy Days). Two additional 
variables — Children in Poverty and Low Birthweight — were identified as 
potential mediators for subsequent mediation analysis.


In [17]:
df_select = pd.read_excel(
    file_path,
    sheet_name='Select Measure Data',
    header=[0, 1]
)

print('Shape:', df_select.shape)
print('\nColumn names:')
for col in df_select.columns:
    print(col)

Shape: (59, 272)

Column names:
('Unnamed: 0_level_0', 'FIPS')
('Unnamed: 1_level_0', 'State')
('Unnamed: 2_level_0', 'County')
('Premature Death', 'Unreliable')
('Premature Death', 'Deaths')
('Premature Death', 'Years of Potential Life Lost Rate')
('Premature Death', '95% CI - Low')
('Premature Death', '95% CI - High')
('Premature Death', 'National Z-Score')
('Premature Death', 'YPLL Rate (Hispanic (all races))')
('Premature Death', 'YPLL Rate (Hispanic (all races)) 95% CI - Low')
('Premature Death', 'YPLL Rate (Hispanic (all races)) 95% CI - High')
('Premature Death', 'YPLL Rate (Hispanic (all races)) Unreliable')
('Premature Death', 'YPLL Rate (Non-Hispanic AIAN)')
('Premature Death', 'YPLL Rate (Non-Hispanic AIAN) 95% CI - Low')
('Premature Death', 'YPLL Rate (Non-Hispanic AIAN) 95% CI - High')
('Premature Death', 'YPLL Rate (Non-Hispanic AIAN) Unreliable')
('Premature Death', 'YPLL Rate (Non-Hispanic Asian)')
('Premature Death', 'YPLL Rate (Non-Hispanic Asian) 95% CI - Low')
('Pre

In [18]:
df_select_outcomes = df_select[[('Unnamed: 0_level_0', 'FIPS'),('Unnamed: 2_level_0', 'County'),
('Premature Death', 'Years of Potential Life Lost Rate'), ('Poor Mental Health Days', 'Average Number of Mentally Unhealthy Days'),
('Preventable Hospital Stays', 'Preventable Hospitalization Rate')]]

print(df_select_outcomes.head(3).to_string())

  Unnamed: 0_level_0 Unnamed: 2_level_0                   Premature Death                   Poor Mental Health Days       Preventable Hospital Stays
                FIPS             County Years of Potential Life Lost Rate Average Number of Mentally Unhealthy Days Preventable Hospitalization Rate
0               6000                NaN                       6373.194549                                  4.650926                             2153
1               6001            Alameda                       4981.213394                                  5.070780                             2102
2               6003             Alpine                               NaN                                  5.541764                              921


In [19]:
df_select_mediators = df_select[[('Unnamed: 0_level_0', 'FIPS'),('Unnamed: 2_level_0', 'County'), ('Children in Poverty', '% Children in Poverty'),('Low Birthweight', '% Low Birthweight')]]

print(df_select_mediators)
print(df_select_mediators.head(3).to_string())


   Unnamed: 0_level_0 Unnamed: 2_level_0   Children in Poverty  \
                 FIPS             County % Children in Poverty   
0                6000                NaN                  15.3   
1                6001            Alameda                  10.3   
2                6003             Alpine                  26.6   
3                6005             Amador                  12.0   
4                6007              Butte                  18.7   
5                6009          Calaveras                  16.2   
6                6011             Colusa                  13.7   
7                6013       Contra Costa                  10.0   
8                6015          Del Norte                  22.5   
9                6017          El Dorado                   6.5   
10               6019             Fresno                  25.1   
11               6021              Glenn                  19.2   
12               6023           Humboldt                  20.5   
13        

In [20]:
# drop the state row
df_select_outcomes = df_select_outcomes.drop(index=0)
df_select_mediators = df_select_mediators.drop(index=0)

# check for null values
print(df_select_outcomes.isna().sum())
print(df_select_mediators.isna().sum())

Unnamed: 0_level_0          FIPS                                         0
Unnamed: 2_level_0          County                                       0
Premature Death             Years of Potential Life Lost Rate            2
Poor Mental Health Days     Average Number of Mentally Unhealthy Days    0
Preventable Hospital Stays  Preventable Hospitalization Rate             0
dtype: int64
Unnamed: 0_level_0   FIPS                     0
Unnamed: 2_level_0   County                   0
Children in Poverty  % Children in Poverty    0
Low Birthweight      % Low Birthweight        1
dtype: int64


In [21]:
# check counties impacted by nulls
df_select_outcomes[df_select_outcomes[('Premature Death', 'Years of Potential Life Lost Rate')].isna()]



,Unnamed: 0_level_0,Unnamed: 2_level_0,Premature Death,Poor Mental Health Days,Preventable Hospital Stays
,FIPS,County,Years of Potential Life Lost Rate,Average Number of Mentally Unhealthy Days,Preventable Hospitalization Rate
2,6003,Alpine,NaN,5.541764,921
46,6091,Sierra,NaN,5.482310,1644


In [22]:
df_select_mediators[df_select_mediators[('Low Birthweight', '% Low Birthweight')].isna()]

,Unnamed: 0_level_0,Unnamed: 2_level_0,Children in Poverty,Low Birthweight
,FIPS,County,% Children in Poverty,% Low Birthweight
2,6003,Alpine,26.6,NaN


Null values were observed in two rural counties: Alpine (FIPS 06003) and 
Sierra (FIPS 06091). All county data will be retained in order to preserve 
valid observations for other outcome variables. Null values will be excluded 
listwise at the model level rather than removed from the dataset. This 
approach acknowledges unavoidable data suppression in low-population counties 
as a limitation of the analysis.

In [ ]:
# revisit after column naming conventions are understood


# # relevent_variables = ['Unnamed: 0_level_0', 'FIPS','Unnamed: 2_level_0', 'County', 'Premature Death', 'Years of Potential Life Lost Rate', 'Poor Mental Health Days', 'Average Number of Mentally Unhealthy Days',
# # 'Preventable Hospital Stays', 'Preventable Hospitalization Rate','Children in Poverty', '% Children in Poverty','Low Birthweight', '% Low Birthweight']
   
# for f in files:
#     if '2025' in f:
#         continue
#     file_path = '../data/raw/county_health_rankings/' + f
#     df_year = pd.read_excel(
#         file_path,
#         sheet_name='Select Measure Data',
#         header=[0, 1])
#     print('File:', f, '| Shape:', df_year.shape)     



ValueError: Worksheet named 'Select Measure Data' not found

In [ ]:
for f in files:
    if '2025' in f:
        continue
    file_path = '../data/raw/county_health_rankings/' + f
    xl = pd.ExcelFile(file_path)
    print(f, '|', xl.sheet_names)

2023 County Health Rankings California Data - v3.xlsx | ['Introduction', 'Outcomes & Factors Rankings', 'Outcomes & Factors SubRankings', 'Ranked Measure Data', 'Additional Measure Data', 'Ranked Measure Sources & Years', 'Addtl Measure Sources & Years']
2018 County Health Rankings California Data - v3.xls | ['Introduction', 'Outcomes & Factors Rankings', 'Outcomes & Factors SubRankings', 'Ranked Measure Data', 'Additional Measure Data', 'Ranked Measure Sources & Years', 'Addtl Measure Sources & Years']
2017 County Health Rankings California Data - v2.xls | ['Introduction', 'Outcomes & Factors Rankings', 'Outcomes & Factors SubRankings', 'Ranked Measure Data', 'Additional Measure Data', 'Ranked Measure Sources & Years', 'Addtl Measure Sources & Years']
2020 County Health Rankings California Data - v1_0.xlsx | ['Introduction', 'Outcomes & Factors Rankings', 'Outcomes & Factors SubRankings', 'Ranked Measure Data', 'Additional Measure Data', 'Ranked Measure Sources & Years', 'Addtl Measur

In [12]:
# Load 2016 CHR file
file_path = '../data/raw/county_health_rankings/2016 County Health Rankings California Data - v3.xls'

df_2016 = pd.read_excel(
    file_path,
    sheet_name='Ranked Measure Data',
    header=[0, 1]  # Multi-row header based on what we saw
)

# Inspect
print('Shape:', df_2016.shape)
print('\nColumn names:')
for col in df_2016.columns:
    print(col)

Shape: (61, 156)

Column names:
('Unnamed: 0_level_0', 'FIPS')
('Unnamed: 1_level_0', 'State')
('Unnamed: 2_level_0', 'County')
('Premature death', '# Deaths')
('Premature death', 'Years of Potential Life Lost Rate')
('Premature death', '95% CI - Low')
('Premature death', '95% CI - High')
('Premature death', 'Z-Score')
('Poor or fair health', '% Fair/Poor')
('Poor or fair health', '95% CI - Low')
('Poor or fair health', '95% CI - High')
('Poor or fair health', 'Z-Score')
('Poor physical health days', 'Physically Unhealthy Days')
('Poor physical health days', '95% CI - Low')
('Poor physical health days', '95% CI - High')
('Poor physical health days', 'Z-Score')
('Poor mental health days', 'Mentally Unhealthy Days')
('Poor mental health days', '95% CI - Low')
('Poor mental health days', '95% CI - High')
('Poor mental health days', 'Z-Score')
('Low birthweight', 'Unreliable')
('Low birthweight', '# Low Birthweight Births')
('Low birthweight', '# Live Births')
('Low birthweight', '% LBW')


In [13]:
for f in files:
    if '2025' in f:
        continue
    file_path = '../data/raw/county_health_rankings/' + f
    xl = pd.ExcelFile(file_path)
    sheet = 'Select Measure Data' if '2024' in f else 'Ranked Measure Data'
    df_year = pd.read_excel(file_path, sheet_name=sheet, header=[0,1])
    unique_cols = []
    for col in df_year.columns:
        if col[0] not in unique_cols:
            unique_cols.append(col[0])
    print(f, '|', unique_cols)


2023 County Health Rankings California Data - v3.xlsx | ['Unnamed: 0_level_0', 'Unnamed: 1_level_0', 'Unnamed: 2_level_0', 'Premature Death', 'Poor or Fair Health', 'Poor Physical Health Days', 'Poor Mental Health Days', 'Low Birthweight', 'Adult Smoking', 'Adult Obesity', 'Food Environment Index', 'Physical Inactivity', 'Access to Exercise Opportunities', 'Excessive Drinking', 'Alcohol-Impaired Driving Deaths', 'Sexually Transmitted Infections', 'Teen Births', 'Uninsured', 'Primary Care Physicians', 'Dentists', 'Mental Health Providers', 'Preventable Hospital Stays', 'Mammography Screening', 'Flu Vaccinations', 'High School Completion', 'Some College', 'Unemployment', 'Children in Poverty', 'Income Inequality', 'Children in Single-Parent Households', 'Social Associations', 'Injury Deaths', 'Air Pollution - Particulate Matter', 'Drinking Water Violations', 'Severe Housing Problems', 'Driving Alone to Work', 'Long Commute - Driving Alone']
2018 County Health Rankings California Data - v